# Create MITRE ATT&CK Tactics & Techniques json files

Download MITRE ATT&CK taxonomy data (most recent **enterprise-attack**, **ics-attack** & **mobile-attack** json files) from: https://github.com/mitre-attack/attack-stix-data.

Make sure the downloaded taxonomy data files are available in the mitre_attack_files folder.

In [81]:
from mitreattack.stix20 import MitreAttackData
from collections import defaultdict
import json

In [64]:
def get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique_stix_id):

    # get data components detecting technique
    datacomponents_detects_technique = mitre_attack_data.get_datacomponents_detecting_technique(technique_stix_id)

    # Get all data sources and data components.
    data_sources = []
    data_components = []
    for d in datacomponents_detects_technique:
        datacomponent = d["object"]
        datasource = mitre_attack_data.get_object_by_stix_id(datacomponent.x_mitre_data_source_ref)
        data_sources.append(datasource.name)
        data_components.append(datacomponent.name)

    # Remove duplicates in the data sources.
    data_sources = list(set(data_sources))

    return data_sources, data_components

In [137]:
global temp
temp = []

In [196]:
def process_mitre_attack_json(json_file_path):
    """Process the ATT&CK data from a single ATT&CK json file."""
    
    # Initialize the MitreAttackData class
    mitre_attack_data = MitreAttackData(json_file_path)
    
    # Fetch all tactics
    tactics = mitre_attack_data.get_tactics_by_matrix()
    
    # Prepare the structure for the JSON
    tactics_entries = []

    # Create a list of all platforms (will have many duplicates)
    all_platforms = []
    
    # We assume that our ATT&CK JSON files contain a single domain, which we extract here
    if 'Enterprise ATT&CK' in list(tactics.keys()):
        domain_str = 'Enterprise ATT&CK'
        domain = 'enterprise-attack'
    # We ignore the 'Network-Based Effects' key that appears in the tactics.keys() for the Mobile domain
    if 'Mobile ATT&CK' in list(tactics.keys()):
        domain_str = 'Mobile ATT&CK'
        domain = 'mobile-attack'
    if 'ATT&CK for ICS' in list(tactics.keys()):
        domain_str = 'ATT&CK for ICS'
        domain = 'ics-attack'
    


    #### Iterate over all tactics
    for tactic in tactics[domain_str]:
        # Fetch all techniques for the current tactic
        techniques = mitre_attack_data.get_techniques_by_tactic(tactic['x_mitre_shortname'], domain, remove_revoked_deprecated=True)

        # Prepare the techniques list with sub-techniques nested under their parent techniques
        techniques_dict = {}
        for technique in techniques:
            
            # # Process data sources strings into separate 'data sources' and 'data components' labels.
            # data_sources_string = technique.get('x_mitre_data_sources')
            # data_sources = []
            # data_components = []
            # if data_sources_string != None:
            #     for x in data_sources_string:
            #         data_source, data_component = x.split(': ')
            #         data_sources.append(data_source)
            #         data_components.append(data_component)
            #     data_sources = list(set(data_sources))
            #     data_components = list(set(data_components))

            platforms = technique.get('x_mitre_platforms')
            if platforms != None:
                all_platforms.extend(platforms)
            data_sources, data_components = get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique.id)
            groups = mitre_attack_data.get_groups_using_technique(technique.id)
            groups = [x['object']['name'] for x in groups]
            occurrence_groups = len(groups)

            software = mitre_attack_data.get_software_using_technique(technique.id)
            software = [x['object']['name'] for x in software]
            occurrence_software = len(software)
            
            technique_entry = {
                "name": technique['name'],
                "external_id": technique['external_references'][0]['external_id'],
                "platforms": platforms,
                'groups': groups,
                'occurrence_groups': occurrence_groups,
                'software': software,
                'occurrence_software': occurrence_software,
                'occurrence_total': occurrence_groups + occurrence_software,
                'data_sources': data_sources,
                'data_components': data_components,
                "visibility": False,  # Set default visibility to False, can be modified as needed.
                "visibility_ratio": 0,
            }
            if technique.get('x_mitre_is_subtechnique'):
                parent_id = technique['external_references'][0]['external_id'].split('.')[0]
                if parent_id in techniques_dict:
                    if 'sub_techniques' not in techniques_dict[parent_id]:
                        techniques_dict[parent_id]['sub_techniques'] = []
                    techniques_dict[parent_id]['sub_techniques'].append(technique_entry)
                else:
                    techniques_dict[parent_id] = {
                        "sub_techniques": [technique_entry]
                    }
            else:
                techniques_dict[technique_entry['external_id']] = technique_entry
    
        # Sort the techniques alphabetically by name
        techniques_list = sorted([value for value in techniques_dict.values() if 'name' in value], key=lambda x: x['name'])
    
        # Sort the sub-techniques alphabetically by name
        for technique in techniques_list:
            if 'sub_techniques' in technique:
                technique['sub_techniques'] = sorted(technique['sub_techniques'], key=lambda x: x['name'])

        # Compute (sub)technique counts for the tactic.
        technique_count = len(techniques_list)
        subtechnique_count = sum([len(x['sub_techniques']) for x in techniques_list if 'sub_techniques' in x])
        all_technique_count = technique_count + subtechnique_count

        # Prepare the tactic entry
        tactic_entry = {
            "name": tactic['name'],
            "external_id": tactic['external_references'][0]['external_id'],
            "techniques": techniques_list,
            "technique_count": technique_count,
            "subtechnique_count": subtechnique_count,
            "all_technique_count": all_technique_count,
        }
        
        tactics_entries.append(tactic_entry)

    
    #### Pre-process the tactics_entries, so we can compute an order for the occurrence.
    total_techniques_count = 0
    for tactic_entry in tactics_entries:
        total_techniques_count += len(tactic_entry['techniques'])

    # Put all techniques in a single list.
    all_techniques = []
    for tactic_entry in tactics_entries:
        all_techniques += tactic_entry['techniques']

    # Assign an occurrence value based on a techniques occurrence frequency.
    # We gather all the frequencies, and perform normalization based on the position of the frequency
    # value in the total sorted list of frequency values. The normalized values are assigned to techniques.
    def assign_normalized_frequencies(occurrence_key):
        frequency_dict = defaultdict(list)
        for technique in all_techniques:
            freq_key = technique[occurrence_key]
            frequency_dict[freq_key].append(technique)

        sorted_frequency_dict = dict(sorted(frequency_dict.items()))

        # Assign the normalized frequency value to each technique.
        for freq, techniques in sorted_frequency_dict.items():
            normalized_freq_value = freq / len(sorted_frequency_dict)
            for technique in techniques:
                technique[f'{occurrence_key}_order_normalized'] = normalized_freq_value

    # Apply the function for each occurrence key.
    assign_normalized_frequencies('occurrence_total')
    assign_normalized_frequencies('occurrence_groups')
    assign_normalized_frequencies('occurrence_software')
    
    
    ### Create platforms entries
    platforms_entries = []
    
    # Remove duplicates in all_platforms list
    all_platforms = list(set(all_platforms))

    # Loop over all platforms
    for platform in all_platforms:
        techniques_by_platform = mitre_attack_data.get_techniques_by_platform(platform, remove_revoked_deprecated=True)
        all_techniques = [x.name for x in techniques_by_platform]
        techniques = [x.name  for x in techniques_by_platform if x.x_mitre_is_subtechnique == False]
        subtechniques = [x.name  for x in techniques_by_platform if x.x_mitre_is_subtechnique == True]

        # techniques_used_by_group[0]['object'].x_mitre_is_subtechnique

        # Prepare the platform entry
        platform_entry = {
            "name": platform,
            "active_in_filter": True,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        platforms_entries.append(platform_entry)

    # Sort platform entries by technique_count
    platforms_entries = sorted(platforms_entries, key=lambda item: item['all_technique_count'], reverse=True)


    
    #### Create groups entries
    groups_entries = []

    # Get all groups
    groups = mitre_attack_data.get_groups(remove_revoked_deprecated=True)

    # Loop over all groups
    for group in groups:
        techniques_used_by_group = mitre_attack_data.get_techniques_used_by_group(group.id)
        all_techniques = [x['object'].name for x in techniques_used_by_group]
        techniques = [x['object'].name for x in techniques_used_by_group if x['object'].x_mitre_is_subtechnique == False]
        subtechniques = [x['object'].name for x in techniques_used_by_group if x['object'].x_mitre_is_subtechnique == True]

        # Prepare group entry
        group_entry = {
            "name": group.name,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        groups_entries.append(group_entry)

    # Sort group entries by technique_count
    groups_entries = sorted(groups_entries, key=lambda item: item['all_technique_count'], reverse=True)


    
    #### Create software entries
    software_entries = []

    # Get all softwares
    softwares = mitre_attack_data.get_software(remove_revoked_deprecated=True)

    # Loop over all softwares
    for software in softwares:
        techniques_used_by_software = mitre_attack_data.get_techniques_used_by_software(software.id)
        all_techniques = [x['object'].name for x in techniques_used_by_software]
        techniques = [x['object'].name for x in techniques_used_by_software if x['object'].x_mitre_is_subtechnique == False]
        subtechniques = [x['object'].name for x in techniques_used_by_software if x['object'].x_mitre_is_subtechnique == True]

        # Prepare software entry
        software_entry = {
            "name": software.name,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        software_entries.append(software_entry)

    # Sort software entries by technique_count
    software_entries = sorted(software_entries, key=lambda item: item['all_technique_count'], reverse=True)
    

    
    #### Create datasources entries
    datasources_entries = []

    # Get all data sources
    datasources = mitre_attack_data.get_datasources(remove_revoked_deprecated=True)

    # Loop over all datasources
    for datasource in datasources:

        # Prepare datasource entry
        datasource_entry = {
            'name': datasource['name'],
            'active_in_filter': True,
        }
    
        datasources_entries.append(datasource_entry)


    
    #### Create datacomponent entries
    datacomponents_entries = []

    # Get all data components
    datacomponents = mitre_attack_data.get_datacomponents(remove_revoked_deprecated=True)

    # Loop over all datacomponens
    for datacomponent in datacomponents:

        # Get all techniques detected by each datacomponent
        techniques_detected_by_datacomponent = mitre_attack_data.get_techniques_detected_by_datacomponent(datacomponent['id'])

        # Prepare datacomponent entry
        datacomponent_entry = {
            'name': datacomponent['name'],
            'active_in_filter': True,
            'detected_techniques': [x['object']['name'] for x in techniques_detected_by_datacomponent],
            'visibility': False,
            'quality': {
                'device_completeness': None,
                'data_field_completeness': None,
                'timeliness': None,
                'consistency': None,
                'retention': None,
            }
        }
    
        datacomponents_entries.append(datacomponent_entry)


    
    #### Combine all data for the domain into a single entry.
    domain_entry = {
        "tactics": tactics_entries,
        "platforms": platforms_entries,
        "data_sources": datasources_entries,
        "data_components": datacomponents_entries,
        "groups": groups_entries,
        "softwares": software_entries,
    }

    return domain_entry

In [197]:
def flatten(lst):
    """
    Flattens a nested list, ignoring NoneType objects.

    Parameters:
    lst (list): A list that may contain nested lists and NoneType objects.

    Returns:
    list: A flattened list with NoneType objects removed.
    """
    result = []

    def _flatten(sublist):
        for item in sublist:
            if item is None:
                continue
            if isinstance(item, list):
                _flatten(item)
            else:
                result.append(item)

    _flatten(lst)
    return result

In [198]:
def convert_list(attribute_list):
    return [{"name": name, "active_in_filter": active_in_filter} for name, active_in_filter in list(zip(attribute_list, [True]*len(attribute_list)))]

In [199]:
# Load JSON files of all ATT&CK domains.
enterprise = process_mitre_attack_json("mitre_attack_files/enterprise-attack-16.1.json")
ics = process_mitre_attack_json("mitre_attack_files/ics-attack-16.1.json")
mobile = process_mitre_attack_json("mitre_attack_files/mobile-attack-16.1.json")

# Prepare the final JSON structure.
final_json = {
    "enterprise": enterprise,
    "ics": ics,
    "mobile": mobile,
}
    
# Write the JSON to a file.
with open("tactics_and_techniques_by_domain.json", "w") as json_file:
    json.dump(final_json, json_file, indent=2)

print("JSON file created successfully!")

JSON file created successfully!


In [175]:
temp

all_techniques = [print(list(x['object'].keys())) for x in temp]

['type', 'spec_version', 'id', 'created_by_ref', 'created', 'modified', 'name', 'description', 'kill_chain_phases', 'revoked', 'external_references', 'object_marking_refs', 'x_mitre_attack_spec_version', 'x_mitre_contributors', 'x_mitre_data_sources', 'x_mitre_deprecated', 'x_mitre_detection', 'x_mitre_domains', 'x_mitre_impact_type', 'x_mitre_is_subtechnique', 'x_mitre_modified_by_ref', 'x_mitre_platforms', 'x_mitre_version']
['type', 'spec_version', 'id', 'created_by_ref', 'created', 'modified', 'name', 'description', 'kill_chain_phases', 'revoked', 'external_references', 'object_marking_refs', 'x_mitre_attack_spec_version', 'x_mitre_contributors', 'x_mitre_data_sources', 'x_mitre_deprecated', 'x_mitre_detection', 'x_mitre_domains', 'x_mitre_is_subtechnique', 'x_mitre_modified_by_ref', 'x_mitre_platforms', 'x_mitre_version']
['type', 'spec_version', 'id', 'created_by_ref', 'created', 'modified', 'name', 'description', 'kill_chain_phases', 'revoked', 'external_references', 'object_mar

In [148]:
temp[0].x_mitre_is_subtechnique

False

In [146]:
list(temp[0].keys())

['type',
 'spec_version',
 'id',
 'created_by_ref',
 'created',
 'modified',
 'name',
 'description',
 'kill_chain_phases',
 'revoked',
 'external_references',
 'object_marking_refs',
 'x_mitre_attack_spec_version',
 'x_mitre_contributors',
 'x_mitre_data_sources',
 'x_mitre_deprecated',
 'x_mitre_detection',
 'x_mitre_domains',
 'x_mitre_is_subtechnique',
 'x_mitre_modified_by_ref',
 'x_mitre_platforms',
 'x_mitre_version']

In [119]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")
techniques_used_by_group = mitre_attack_data.get_techniques_used_by_group("intrusion-set--00f67a77-86a4-4adf-be26-1a54fc713340")

In [130]:
techniques_used_by_group[0]['object']['name']

'Data Encrypted for Impact'

In [117]:
techniques_used_by_group[0]['object'].x_mitre_is_subtechnique

False

In [ ]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")
techniques_by_platform = mitre_attack_data.get_techniques_by_platform('Windows', remove_revoked_deprecated=True)

In [87]:
list(techniques_by_platform[0].keys())

['type',
 'spec_version',
 'id',
 'created_by_ref',
 'created',
 'modified',
 'name',
 'description',
 'kill_chain_phases',
 'revoked',
 'external_references',
 'object_marking_refs',
 'x_mitre_attack_spec_version',
 'x_mitre_data_sources',
 'x_mitre_defense_bypassed',
 'x_mitre_detection',
 'x_mitre_domains',
 'x_mitre_is_subtechnique',
 'x_mitre_modified_by_ref',
 'x_mitre_platforms',
 'x_mitre_version']

In [88]:
techniques_by_platform[0].x_mitre_is_subtechnique

True

In [92]:
len([x.name for x in techniques_by_platform if x.x_mitre_is_subtechnique == False])

163

In [ ]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")

# Fetch all tactics
tactics = mitre_attack_data.get_tactics_by_matrix()

#### Iterate over all tactics
for tactic in tactics[ 'Enterprise ATT&CK']:
    # Fetch all techniques for the current tactic
    techniques = mitre_attack_data.get_techniques_by_tactic(tactic['x_mitre_shortname'], 'enterprise-attack', remove_revoked_deprecated=True)


In [14]:
list(techniques[0].keys())

['type',
 'spec_version',
 'id',
 'created_by_ref',
 'created',
 'modified',
 'name',
 'description',
 'kill_chain_phases',
 'revoked',
 'external_references',
 'object_marking_refs',
 'x_mitre_attack_spec_version',
 'x_mitre_contributors',
 'x_mitre_data_sources',
 'x_mitre_deprecated',
 'x_mitre_detection',
 'x_mitre_domains',
 'x_mitre_impact_type',
 'x_mitre_is_subtechnique',
 'x_mitre_modified_by_ref',
 'x_mitre_platforms',
 'x_mitre_version']

In [22]:
techniques[0]['external_references'][0]['external_id']

'T1561.002'

In [24]:
techniques[0].get('name')

'Disk Structure Wipe'

In [ ]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")

# get software related to T1014
technique_stix_id = "attack-pattern--0f20e3cb-245b-4a61-8a91-2d93f7cb0e9b"
software_using_t1014 = mitre_attack_data.get_software_using_technique(technique_stix_id)

print(f"Software using T1014 ({len(software_using_t1014)}):")
for s in software_using_t1014:
    software = s["object"]
    print(f"* {software.name} ({mitre_attack_data.get_attack_id(software.id)})")

Software using T1014 (21):
* HTRAN (S0040)
* Ebury (S0377)
* Carberp (S0484)
* Ramsay (S0458)
* Drovorub (S0502)
* HIDEDRV (S0135)
* Skidmap (S0468)
* Umbreon (S0221)
* Stuxnet (S0603)
* COATHANGER (S1105)
* Hacking Team UEFI Rootkit (S0047)
* HiddenWasp (S0394)
* Hildegard (S0601)
* Hikit (S0009)
* Winnti for Linux (S0430)
* Zeroaccess (S0027)
* LoJax (S0397)
* Uroburos (S0022)
* WarzoneRAT (S0670)
* Caterpillar WebShell (S0572)
* PoisonIvy (S0012)


In [7]:
software_using_t1014

[{'object': Tool(type='tool', spec_version='2.1', id='tool--d5e96a35-7b0b-4c6a-9533-d63ecbda563e', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2017-05-31T21:32:32.011Z', modified='2022-04-25T14:00:00.188Z', name='HTRAN', description='[HTRAN](https://attack.mitre.org/software/S0040) is a tool that proxies connections through intermediate hops and aids users in disguising their true geographical location. It can be used by adversaries to hide their location when interacting with the victim networks. (Citation: Operation Quantum Entanglement)(Citation: NCSC Joint Report Public Tools)', revoked=False, external_references=[ExternalReference(source_name='mitre-attack', url='https://attack.mitre.org/software/S0040', external_id='S0040'), ExternalReference(source_name='HUC Packet Transmit Tool', description='(Citation: Operation Quantum Entanglement)'), ExternalReference(source_name='Operation Quantum Entanglement', description='Haq, T., Moran, N., Vashisht, S., S

In [9]:
software = [x['object']['name'] for x in software_using_t1014]
software

['HTRAN',
 'Ebury',
 'Carberp',
 'Ramsay',
 'Drovorub',
 'HIDEDRV',
 'Skidmap',
 'Umbreon',
 'Stuxnet',
 'COATHANGER',
 'Hacking Team UEFI Rootkit',
 'HiddenWasp',
 'Hildegard',
 'Hikit',
 'Winnti for Linux',
 'Zeroaccess',
 'LoJax',
 'Uroburos',
 'WarzoneRAT',
 'Caterpillar WebShell',
 'PoisonIvy']

In [ ]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")

datacomponents = mitre_attack_data.get_datacomponents(remove_revoked_deprecated=True)

print(f"Retrieved {len(datacomponents)} ATT&CK data components.")

Retrieved 106 ATT&CK data components.


In [13]:
list(datacomponents[0].keys())

['type',
 'spec_version',
 'id',
 'created_by_ref',
 'created',
 'modified',
 'revoked',
 'object_marking_refs',
 'name',
 'description',
 'x_mitre_attack_spec_version',
 'x_mitre_data_source_ref',
 'x_mitre_modified_by_ref',
 'x_mitre_version',
 'x_mitre_domains']

In [14]:
datacomponents[0]['id']

'x-mitre-data-component--02d090b6-8157-48da-98a2-517f7edd49fc'

In [15]:
techniques_detected_by_datacomponent = mitre_attack_data.get_techniques_detected_by_datacomponent(datacomponents[0]['id'])

In [21]:
list(techniques_detected_by_datacomponent[0]['object'].keys())

['type',
 'spec_version',
 'id',
 'created_by_ref',
 'created',
 'modified',
 'name',
 'description',
 'kill_chain_phases',
 'revoked',
 'external_references',
 'object_marking_refs',
 'x_mitre_attack_spec_version',
 'x_mitre_contributors',
 'x_mitre_data_sources',
 'x_mitre_detection',
 'x_mitre_domains',
 'x_mitre_is_subtechnique',
 'x_mitre_modified_by_ref',
 'x_mitre_permissions_required',
 'x_mitre_platforms',
 'x_mitre_version']

In [22]:
techniques_detected_by_datacomponent[0]['object']['name']

'Golden Ticket'

In [ ]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")

datasources = mitre_attack_data.get_datasources(remove_revoked_deprecated=True)

print(f"Retrieved {len(datasources)} ATT&CK data sources.")

Retrieved 37 ATT&CK data sources.


In [6]:
list(datasources[0].keys())

['type',
 'spec_version',
 'id',
 'created_by_ref',
 'created',
 'modified',
 'revoked',
 'external_references',
 'object_marking_refs',
 'name',
 'description',
 'x_mitre_attack_spec_version',
 'x_mitre_collection_layers',
 'x_mitre_contributors',
 'x_mitre_domains',
 'x_mitre_modified_by_ref',
 'x_mitre_platforms',
 'x_mitre_version']

In [7]:
datasources[0]['name']

'Pod'

In [ ]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")

groups = mitre_attack_data.get_groups(remove_revoked_deprecated=True)
techniques_used_by_group = mitre_attack_data.get_techniques_used_by_group(groups[0].id)

print(f"Retrieved {len(groups)} ATT&CK groups.")

Retrieved 149 ATT&CK groups.


In [ ]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")

techniques = mitre_attack_data.get_techniques_by_platform("Windows", remove_revoked_deprecated=True)

print(f"There are {len(techniques)} techniques in the Windows platform.")

In [34]:
techniques[0].name

'Extra Window Memory Injection'

In [30]:
list(techniques[0].keys())

['type',
 'spec_version',
 'id',
 'created_by_ref',
 'created',
 'modified',
 'name',
 'description',
 'kill_chain_phases',
 'revoked',
 'external_references',
 'object_marking_refs',
 'x_mitre_attack_spec_version',
 'x_mitre_data_sources',
 'x_mitre_defense_bypassed',
 'x_mitre_detection',
 'x_mitre_domains',
 'x_mitre_is_subtechnique',
 'x_mitre_modified_by_ref',
 'x_mitre_platforms',
 'x_mitre_version']

In [ ]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-16.1.json")

# get groups related to T1014
technique_stix_id = "attack-pattern--0f20e3cb-245b-4a61-8a91-2d93f7cb0e9b"
groups_using_t1014 = mitre_attack_data.get_groups_using_technique(technique_stix_id)

print(f"Groups using T1014 ({len(groups_using_t1014)}):")
for g in groups_using_t1014:
    group = g["object"]
    print(f"* {group.name} ({mitre_attack_data.get_attack_id(group.id)})")

Groups using T1014 (5):
* Winnti Group (G0044)
* APT41 (G0096)
* Rocke (G0106)
* TeamTNT (G0139)
* APT28 (G0007)


In [8]:
test = [{"name": name, "show": show} for name, show in list(zip(ics_platforms, [True]*len(enterprise_platforms)))]
test

[{'name': 'None', 'show': True}]

In [9]:
list(zip(enterprise_platforms, [True]*len(enterprise_platforms)))

[('Azure AD', True),
 ('Containers', True),
 ('macOS', True),
 ('Linux', True),
 ('Network', True),
 ('Google Workspace', True),
 ('SaaS', True),
 ('Office 365', True),
 ('PRE', True),
 ('Windows', True),
 ('IaaS', True)]